In [1]:
# # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # %reload_ext autotime  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution  # Disabled for non-interactive execution
import pandas as pd
import requests
import geopandas as gpd
from tqdm.auto import tqdm
from tqdm.contrib.concurrent import process_map
import time
import os
from glob import glob
from coastsat import SDS_transects
import json
import matplotlib.pyplot as plt
import dotenv
dotenv.load_dotenv()

False

In [2]:
poly = gpd.read_file("polygons.geojson")
poly = poly[poly.id.str.startswith("nzd")]
poly.set_index("id", inplace=True)
poly

,area,id_sorted,northing,geometry
id,,,,
nzd0001,3.069093e+06,nzd0001,-4.085604e+06,"POLYGON ((172.96406 -34.43054, 172.99324 -34.4..."


In [3]:
# Test cell - skip if tides.csv exists or no API key
sitename = "nzd0001"
if os.path.isfile(f"data/{sitename}/tides.csv"):
    print(f"tides.csv already exists for {sitename}, skipping API test")
elif os.environ.get("NIWA_API_KEY"):
    dates = pd.to_datetime(pd.read_csv(f"data/{sitename}/transect_time_series.csv").dates).dt.round("10min")
    point = poly.geometry[sitename].centroid
    datetime = dates.iloc[0]
    print(datetime, point)
    try:
        r = requests.get("https://api.niwa.co.nz/tides/data", params={
            "lat": point.y,
            "long": point.x,
            "numberOfDays": 2,
            "startDate": str(datetime.date()),
            "datum": "MSL",
            "interval": 10,
            "apikey": os.environ["NIWA_API_KEY"]
        }, timeout=(30,30))
        if r.status_code == 200 and "values" in r.json():
            df = pd.DataFrame(r.json()["values"])
            df.index = pd.to_datetime(df.time)
            ax = df.plot(style="o-")
            df[df.index == datetime].plot(color="red", style="x", ax=ax, mew=2, ms=10)
        else:
            print(f"API call failed: {r.status_code}")
    except Exception as e:
        print(f"API test error: {e}")
else:
    print("No API key available and tides.csv doesn't exist - skipping test")


tides.csv already exists for nzd0001, skipping API test


In [4]:
datetime, datetime.tz_convert("Pacific/Auckland")

NameError: name 'datetime' is not defined

When I asked for 1 day (1999-08-17), I got 1999-08-16 12:00 - 1999-08-17 12:00. Have to request 2 days, then pull out the one datetime I want

In [5]:
files = pd.DataFrame({"filename": sorted(glob("data/nzd*/transect_time_series.csv"))})
files["sitename"] = files.filename.str.split("/").str[1]
files["have_tides"] = files.sitename.apply(lambda s: os.path.isfile(f"data/{s}/tides.csv"))
files

,filename,sitename,have_tides
0,data/nzd0001/transect_time_series.csv,nzd0001,True


In [6]:
def get_tide_for_dt(point, datetime):
    max_retries = 10
    retry_count = 0
    while retry_count < max_retries:
        retry_count += 1
        try:
            r = requests.get("https://api.niwa.co.nz/tides/data", params={
                "lat": point.y,
                "long": point.x,
                "numberOfDays": 2,
                "startDate": str(datetime.date()),
                "datum": "MSL",
                "interval": 10, # 10 minute resolution
                "apikey": os.environ["NIWA_API_KEY"]
            }, timeout=(30,30))
        except Exception as e:
            print(e)
            time.sleep(5)
            continue
        if r.status_code == 200:
            df = pd.DataFrame(r.json()["values"])
            df.index = pd.to_datetime(df.time)
            return df.value[datetime]
        elif r.status_code == 429:
            sleep_seconds = 30
            # sleep for x seconds to refresh the count
            print(f'Num of API reqs exceeded, Sleeping for: {sleep_seconds} seconds...')
            time.sleep(sleep_seconds)

for sitename in tqdm(files[~files.have_tides].sitename):
    dates = pd.to_datetime(pd.read_csv(f"data/{sitename}/transect_time_series.csv").dates).dt.round("10min")
    point = poly.geometry[sitename].centroid

    results = []
    for date in tqdm(dates):
        result = get_tide_for_dt(point, date)
        results.append({
            "dates": date,
            "tide": result
        })
    df = pd.DataFrame(results)
    df.set_index("dates", inplace=True)
    df.to_csv(f"data/{sitename}/tides.csv")

0it [00:00, ?it/s]

In [7]:
files["have_tides"] = files.sitename.apply(lambda s: os.path.isfile(f"data/{s}/tides.csv"))

In [8]:
# Transects, origin is landward. Has beach_slope
transects = gpd.read_file("transects_extended.geojson").to_crs(2193).drop_duplicates(subset="id")
transects.set_index("id", inplace=True)
transects

,site_id,orientation,along_dist,along_dist_norm,beach_slope,cil,ciu,trend,n_points,n_points_nonan,r2_score,mae,mse,rmse,intercept,ERODIBILITY,geometry
id,,,,,,,,,,,,,,,,,
aus0001-0000,aus0001,104.347648,0.000000,0.000000,0.085,0.0545,0.2000,-1.441081,767.0,428.0,0.168420,28.102591,1263.560863,35.546601,179.085729,None,"LINESTRING (-422245.836 7118667.88, -421827.54..."
aus0001-0001,aus0001,93.495734,98.408334,0.002935,0.050,0.0387,0.0640,-1.037105,767.0,569.0,0.097874,25.419324,1033.770813,32.152306,212.247788,None,"LINESTRING (-422256.313 7118525.222, -421837.6..."
aus0001-0002,aus0001,82.069341,198.408334,0.005918,0.050,0.0428,0.0647,-0.680019,767.0,588.0,0.053927,22.632907,838.007507,28.948359,205.106151,None,"LINESTRING (-422219.773 7118383.012, -421816.8..."
aus0001-0003,aus0001,81.192757,298.402523,0.008900,0.055,0.0480,0.0659,-0.405198,767.0,598.0,0.023412,20.749758,698.653187,26.432048,191.745881,None,"LINESTRING (-422187.543 7118279.615, -421786.5..."
aus0001-0004,aus0001,81.065473,398.402523,0.011882,0.075,0.0614,0.0922,-0.090025,767.0,608.0,0.001277,19.889328,655.810616,25.608800,175.092121,None,"LINESTRING (-422155.665 7118178.983, -421754.9..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ber0002-0009,ber0002,NaN,NaN,NaN,0.080,0.0638,0.1065,0.159612,199.0,199.0,0.041461,4.326802,33.378983,5.777455,127.283966,None,"LINESTRING (7337061.509 24464210.895, 7337094...."
ber0002-0010,ber0002,NaN,NaN,NaN,0.085,0.0673,0.1096,0.071946,199.0,197.0,0.010730,4.357300,26.732016,5.170301,128.858980,None,"LINESTRING (7336972.269 24464234.663, 7337116...."
ber0002-0011,ber0002,NaN,NaN,NaN,0.105,0.0797,0.1462,0.081426,199.0,198.0,0.011823,4.779105,31.534469,5.615556,129.347401,None,"LINESTRING (7336938.536 24464246.657, 7337082...."


In [9]:
def despike(chainage, threshold=40):
    chainage = chainage.dropna()
    chainage, dates = SDS_transects.identify_outliers(chainage.tolist(), chainage.index.tolist(), threshold)
    return pd.Series(chainage, index=dates)

def process_sitename(sitename):
    transects_at_site = transects[transects.site_id == sitename]
    assert len(transects_at_site)
    raw_intersects = pd.read_csv(f"data/{sitename}/transect_time_series.csv")#.drop(columns=["Unnamed: 0"])
    sat_times = pd.to_datetime(raw_intersects.dates).dt.round("10min")
    raw_intersects.set_index("dates", inplace=True)
    raw_intersects.index = pd.to_datetime(raw_intersects.index)
    tides = pd.read_csv(f"data/{sitename}/tides.csv")
    tides.set_index("dates", inplace=True)
    tides.index = pd.to_datetime(tides.index)
    tides = tides[tides.index.isin(sat_times)]
    if not all(sat_times.isin(tides.index)):
        dates = sat_times[~sat_times.isin(tides.index)]
        print(f"Fetching missing tides for {len(dates)} dates at {sitename}")
        point = poly.geometry[sitename].centroid
        results = []
        for date in tqdm(dates):
            result = get_tide_for_dt(point, date)
            results.append({
                "dates": date,
                "tide": result
            })
        new_tides = pd.DataFrame(results)
        new_tides.dates = pd.to_datetime(new_tides.dates)
        new_tides.set_index("dates", inplace=True)
        tides = pd.concat([tides, new_tides])
        tides.sort_index(inplace=True)
        tides.to_csv(f"data/{sitename}/tides.csv")
    corrections = tides.tide.apply(lambda tide: tide / transects_at_site.beach_slope.interpolate().bfill().ffill()).set_index(raw_intersects.index)
    corrections.columns = corrections.columns.astype(str)
    tidally_corrected = raw_intersects + corrections
    tidally_corrected = tidally_corrected.drop(columns="satname").apply(despike, axis=0)
    tidally_corrected.index.name = "dates"
    if len(tidally_corrected) == 0:
        print(f"Despike removed all points for {sitename}")
    tidally_corrected["satname"] = raw_intersects.satname
    tidally_corrected.to_csv(f"data/{sitename}/transect_time_series_tidally_corrected.csv")
    return tidally_corrected

_ = process_map(process_sitename, files.sitename)
#process_sitename("nzd0562")

  0%|          | 0/1 [00:00<?, ?it/s]